In [8]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from statsmodels.stats.weightstats import ttest_ind
import scipy.stats as stats
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.stats.api as sms
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import (r2_score, mean_absolute_error, mean_squared_error)

from xgboost import XGBRegressor

In [9]:
predictor_cols = [
    "co_wrk_aux",
    "no2_wrk_aux",
    "no_wrk_aux",
    "o3_wrk_aux",
    "rh",
    "temp"
]

sensor_to_site = {
    "Sensor03": "dpw",
    "Sensor08": "dpw",
    "Sensor14": "dpw",
    "Sensor16": "dpw",
    "Sensor18": "dpw",
    "Sensor21": "dpw",

    "Sensor17": "mjf",

    "Sensor05": "pema",
    "Sensor06": "pema",
    "Sensor07": "pema",
    "Sensor15": "pema",
    "Sensor20": "pema",
    "Sensor24": "pema",

    "Sensor02": "pha",
    "Sensor04": "pha",
    "Sensor10": "pha",
    "Sensor12": "pha",
    "Sensor19": "pha",
    "Sensor22": "pha",
    "Sensor23": "pha",
}

results = []

data_dir = "ML ready NC dpp no2 datasets"

for sensor, site in sensor_to_site.items():

    train_path = f"{data_dir}/{sensor}train.csv"
    test_path = f"{data_dir}/{sensor}test.csv"

    # load data
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    train_df["datetime_utc"] = pd.to_datetime(train_df["datetime_utc"])
    test_df["datetime_utc"] = pd.to_datetime(test_df["datetime_utc"])

    target_col = f"{site}_quant_no2"

    X_train = train_df[predictor_cols]
    y_train = train_df[target_col]

    X_test = test_df[predictor_cols]
    y_test = test_df[target_col]

    model = XGBRegressor(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    train_r2 = r2_score(y_train, y_train_pred)
    train_mae = mean_absolute_error(y_train, y_train_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))

    test_r2 = r2_score(y_test, y_test_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

    results.append({
        "sensor": sensor,
        "site": site,

        "train_r2": train_r2,
        "train_mae": train_mae,
        "train_rmse": train_rmse,

        "test_r2": test_r2,
        "test_mae": test_mae,
        "test_rmse": test_rmse,

        "n_train": len(train_df),
        "n_test": len(test_df)
    })

    print(f"{sensor} ({site}) done")

results_df = pd.DataFrame(results)

results_df_sorted = results_df.sort_values("test_r2", ascending=False)

results_df_sorted

Sensor03 (dpw) done
Sensor08 (dpw) done
Sensor14 (dpw) done
Sensor16 (dpw) done
Sensor18 (dpw) done
Sensor21 (dpw) done
Sensor17 (mjf) done
Sensor05 (pema) done
Sensor06 (pema) done
Sensor07 (pema) done
Sensor15 (pema) done
Sensor20 (pema) done
Sensor24 (pema) done
Sensor02 (pha) done
Sensor04 (pha) done
Sensor10 (pha) done
Sensor12 (pha) done
Sensor19 (pha) done
Sensor22 (pha) done
Sensor23 (pha) done


,sensor,site,train_r2,train_mae,train_rmse,test_r2,test_mae,test_rmse,n_train,n_test
16,Sensor12,pha,0.996313,0.281402,0.389938,0.867415,1.579456,2.375916,521,5613
13,Sensor02,pha,0.996029,0.311173,0.404677,0.859942,1.694013,2.441957,521,5613
6,Sensor17,mjf,0.991364,0.455192,0.600674,0.855830,1.504823,2.150773,520,5606
14,Sensor04,pha,0.993941,0.370060,0.499874,0.853785,1.753495,2.495985,521,5587
18,Sensor22,pha,0.995301,0.287189,0.382378,0.845140,1.663067,2.432492,464,4889
15,Sensor10,pha,0.995871,0.313310,0.418376,0.796689,2.017391,3.010740,460,4763
19,Sensor23,pha,0.994179,0.393038,0.528864,0.720851,2.501967,3.585667,406,4607
10,Sensor15,pema,0.994154,0.372480,0.490123,0.671668,2.460430,3.416580,464,4754
9,Sensor07,pema,0.989030,0.481162,0.662464,0.665607,2.510911,3.451791,521,5197
5,Sensor21,dpw,0.994072,0.289487,0.378737,0.638365,2.032831,2.931096,395,4082
